In [1]:
import pandas as pd
import numpy as np

In [2]:
df_all = pd.read_parquet("all_merged_repaired.parquet")

In [3]:
print(df_all.columns.tolist())

['date', 'security_id', 'report_date', 'ticker', 'company_id', 'announce_date', 'fiscal_year', 'fiscal_quarter', 'too_many_quarters_flag', 'too_few_quarters_flag', 'transition_flag', 'total_assets', 'cash_and_equivalents', 'inventory', 'receivables', 'net_ppe', 'total_assets_dnu_flag', 'cash_and_equivalents_dnu_flag', 'inventory_dnu_flag', 'receivables_dnu_flag', 'net_ppe_dnu_flag', 'total_assets_imputed_flag', 'cash_and_equivalents_imputed_flag', 'inventory_imputed_flag', 'receivables_imputed_flag', 'net_ppe_imputed_flag', 'total_liabilities', 'current_liabilities', 'long_term_debt', 'short_term_debt', 'book_equity', 'common_equity', 'total_liabilities_imputed_flag', 'current_liabilities_imputed_flag', 'long_term_debt_imputed_flag', 'short_term_debt_imputed_flag', 'book_equity_imputed_flag', 'common_equity_imputed_flag', 'total_liabilities_dnu_flag', 'current_liabilities_dnu_flag', 'long_term_debt_dnu_flag', 'short_term_debt_dnu_flag', 'book_equity_dnu_flag', 'common_equity_dnu_flag',

In [5]:
ALL_MERGED_COLUMNS = list(df_all.columns)
# ============ Quarterly Alpha 需要的列(分组) ============

# A. 键 + 元数据 + 分类
KEYS_META = [
    'date','security_id','ticker','company_id','report_date','announce_date',
    'fiscal_year','fiscal_quarter',
    'gics_sector','gics_industry','gics_subindustry',
]
# 季度结构 / PIT 质检旗标(用于剔坏季度/首季)
QUARTER_QC = [
    'too_many_quarters_flag','too_few_quarters_flag','transition_flag',
    'announce_date_filled_flag','end_forwardfill_flag',
]
# B. 季度会计原始变量(26 个,全部需要)
ACCT = [
    'total_assets','cash_and_equivalents','inventory','receivables','net_ppe',
    'total_liabilities','current_liabilities','long_term_debt','short_term_debt',
    'book_equity','common_equity','common_shares_outstanding_mn','dividend_per_share',
    'operating_cash_flow_ytd','capex_ytd','ocf_increment','capex_increment',
    'revenue','cogs','gross_profit','operating_income','interest_expense',
    'income_taxes','net_income','eps_basic','eps_diluted',
]
# C. 季度 consensus(含 statistic_date 快照键 + estimate_high/low)
CONSENSUS = [
    'consensus_statistic_date','consensus_forecast_period_end',
    'consensus_analyst_count','consensus_numup','consensus_numdown',
    'consensus_eps_mean','consensus_eps_median','consensus_estimate_dispersion',
    'consensus_estimate_high','consensus_estimate_low',
]
# C-opt. 队友加的 PIT 辅助列(days_to_period_end / actual_eps_period_end 已删)
CONSENSUS_OPT = [
    'consensus_days_since_actual_eps',
]
# D. Earnings actuals 事件(Surprise_Direction 用;announce_time 已删)
ACTUALS = [
    'actuals_period_end','actuals_announce_date','actuals_actual_eps',
    'actuals_announce_date_missing',  # 非 _flag 结尾,单列
]

# ---- 显式排除:所有 _imputed_flag + actuals_eps_outlier_flag ----
EXCLUDE = {'actuals_eps_outlier_flag'}
def _keep_flag(c):
    return not c.endswith('_imputed_flag') and c not in EXCLUDE

# ---- 质检旗标:按前缀自动收集,只收 quarterly 相关,不碰 rec_/tp_/industry_/sector_ ----
ACCT_FLAGS = [c for c in ALL_MERGED_COLUMNS
              if c.endswith('_flag') and any(c.startswith(v+'_') for v in ACCT) and _keep_flag(c)]
OCF_CAPEX_FLAGS = [c for c in ['ocf_dnu_flag','capex_dnu_flag','ocf_structural_missing_flag',
                   'capex_structural_missing_flag','ocf_imputed_flag','capex_imputed_flag'] if _keep_flag(c)]
CONSENSUS_FLAGS = [c for c in ALL_MERGED_COLUMNS if c.startswith('consensus_') and c.endswith('_flag') and _keep_flag(c)]
ACTUALS_FLAGS  = [c for c in ALL_MERGED_COLUMNS if c.startswith('actuals_') and c.endswith('_flag') and _keep_flag(c)]

QC_FLAGS = QUARTER_QC + sorted(set(ACCT_FLAGS + OCF_CAPEX_FLAGS)) + CONSENSUS_FLAGS + ACTUALS_FLAGS

# ---- 汇总(去重、保序) ----
GROUPS = [('A 键/元数据/分类',KEYS_META),('B 会计',ACCT),('C consensus',CONSENSUS),
          ('C-opt consensus辅助',CONSENSUS_OPT),('D actuals事件',ACTUALS),
          ('E 质检旗标',QC_FLAGS)]

quarterly_cols, seen = [], set()
for _,g in GROUPS:
    for c in g:
        if c not in seen:
            seen.add(c); quarterly_cols.append(c)

# ---- 自检:确认每一列都真实存在于 df_all ----
missing = [c for c in quarterly_cols if c not in ALL_MERGED_COLUMNS]
print("=== 自检 ===")
print("df_all 总列数:", len(ALL_MERGED_COLUMNS))
print("选中列数    :", len(quarterly_cols))
print("不存在的列  :", missing if missing else "无 ✅ 全部存在")
print()
for name,g in GROUPS:
    g2=[c for c in g if c in ALL_MERGED_COLUMNS]
    print(f"[{name}] {len(g2)} 列")
print()
print("=== quarterly_cols ===")
print(quarterly_cols)

=== 自检 ===
df_all 总列数: 188
选中列数    : 88
不存在的列  : 无 ✅ 全部存在

[A 键/元数据/分类] 11 列
[B 会计] 26 列
[C consensus] 10 列
[C-opt consensus辅助] 1 列
[D actuals事件] 4 列
[E 质检旗标] 36 列

=== quarterly_cols ===
['date', 'security_id', 'ticker', 'company_id', 'report_date', 'announce_date', 'fiscal_year', 'fiscal_quarter', 'gics_sector', 'gics_industry', 'gics_subindustry', 'total_assets', 'cash_and_equivalents', 'inventory', 'receivables', 'net_ppe', 'total_liabilities', 'current_liabilities', 'long_term_debt', 'short_term_debt', 'book_equity', 'common_equity', 'common_shares_outstanding_mn', 'dividend_per_share', 'operating_cash_flow_ytd', 'capex_ytd', 'ocf_increment', 'capex_increment', 'revenue', 'cogs', 'gross_profit', 'operating_income', 'interest_expense', 'income_taxes', 'net_income', 'eps_basic', 'eps_diluted', 'consensus_statistic_date', 'consensus_forecast_period_end', 'consensus_analyst_count', 'consensus_numup', 'consensus_numdown', 'consensus_eps_mean', 'consensus_eps_median', 'consensus_estimat

In [6]:
df_quarterly = df_all[quarterly_cols].copy()

In [7]:
df_quarterly.columns.tolist()

['date',
 'security_id',
 'ticker',
 'company_id',
 'report_date',
 'announce_date',
 'fiscal_year',
 'fiscal_quarter',
 'gics_sector',
 'gics_industry',
 'gics_subindustry',
 'total_assets',
 'cash_and_equivalents',
 'inventory',
 'receivables',
 'net_ppe',
 'total_liabilities',
 'current_liabilities',
 'long_term_debt',
 'short_term_debt',
 'book_equity',
 'common_equity',
 'common_shares_outstanding_mn',
 'dividend_per_share',
 'operating_cash_flow_ytd',
 'capex_ytd',
 'ocf_increment',
 'capex_increment',
 'revenue',
 'cogs',
 'gross_profit',
 'operating_income',
 'interest_expense',
 'income_taxes',
 'net_income',
 'eps_basic',
 'eps_diluted',
 'consensus_statistic_date',
 'consensus_forecast_period_end',
 'consensus_analyst_count',
 'consensus_numup',
 'consensus_numdown',
 'consensus_eps_mean',
 'consensus_eps_median',
 'consensus_estimate_dispersion',
 'consensus_estimate_high',
 'consensus_estimate_low',
 'consensus_days_since_actual_eps',
 'actuals_period_end',
 'actuals_annou

In [10]:
"""
按调仓日(每年 3/31, 6/30, 9/30, 12/31)重建 per-stock 面板。
每个 (security_id, rebalance_date) 一行,三块分别对齐:

  FMT(fundamentals) : 按 announce_date          取 <= 调仓日 的最近一条
  actuals           : 按 actuals_announce_date   取 <= 调仓日 的最近一条 -> 得到 (期P, 公布日A)
  consensus         : 取 forecast_period_end == P 且 statistic_date <= A 的最后一条

过期数据守卫(新增):
  merge_asof 会把很久以前的一条也拉过来(中间缺一期财报时会在多个调仓日重复)。
  规则:抓来的公布日必须落在 [上一个调仓日, 本调仓日] 这一个季度窗口内,否则整块置 NaN。
    - FMT:     announce_date         < 上一调仓日 -> FMT 数据置 NaN(保留 ticker/gics 等身份列)
    - actuals: actuals_announce_date  < 上一调仓日 -> actuals 全置 NaN
    - consensus: 对应 actual 的 EPS 为 NaN -> consensus 全置 NaN
"""
import numpy as np
import pandas as pd

REBAL_MD = [(3, 31), (6, 30), (9, 30), (12, 31)]
# 身份/分类列:即使基本面过期也保留(它们不随季度失效)
IDENTITY_KEEP = ['ticker', 'company_id', 'gics_sector', 'gics_industry', 'gics_subindustry']


def build_rebalance_panel(df):
    df = df.sort_values(['security_id', 'date']).reset_index(drop=True)

    # 统一所有 datetime 列精度到 [ns](kind=='M' 覆盖 us/ms/ns)
    for c in df.columns:
        if df[c].dtype.kind == 'M':
            df[c] = df[c].astype('datetime64[ns]')

    # bool 旗标列转 object:bool dtype 存不了 NaN,过期守卫置 NaN 会报错
    for c in df.columns:
        if df[c].dtype == 'bool':
            df[c] = df[c].astype('object')

    consensus_cols = [c for c in df.columns if c.startswith('consensus_')]
    actuals_cols   = [c for c in df.columns if c.startswith('actuals_')]
    fmt_cols       = [c for c in df.columns
                      if c not in consensus_cols + actuals_cols + ['security_id', 'date']]

    # 调仓日网格(强制 [ns]),限制在每只票数据区间内
    yrs = range(df['date'].dt.year.min(), df['date'].dt.year.max() + 1)
    all_rebal = pd.to_datetime([f'{y}-{m:02d}-{d:02d}' for y in yrs
                                for m, d in REBAL_MD]).astype('datetime64[ns]')
    rng = df.groupby('security_id')['date'].agg(['min', 'max'])
    grid = (rng.assign(key=1).reset_index()
              .merge(pd.DataFrame({'rebalance_date': all_rebal, 'key': 1}), on='key')
              .query('min <= rebalance_date <= max')
              [['security_id', 'rebalance_date']]
              .sort_values(['security_id', 'rebalance_date']).reset_index(drop=True))

    # 上一个调仓日(即上一个季末),窗口 = (prev_q, rebalance_date]
    grid['prev_q'] = grid['rebalance_date'] - pd.offsets.QuarterEnd(1)

    # ---------- FMT: announce_date <= 调仓日 最近一条 ----------
    fmt = (df[['security_id'] + fmt_cols]
           .dropna(subset=['announce_date'])
           .sort_values('announce_date')
           .drop_duplicates(['security_id', 'announce_date'], keep='last'))
    res = pd.merge_asof(grid.sort_values('rebalance_date'), fmt.sort_values('announce_date'),
                        by='security_id', left_on='rebalance_date', right_on='announce_date',
                        direction='backward')
    # 守卫:announce_date 早于上一调仓日 -> FMT 数据过期,置 NaN(保留身份/分类列)
    fmt_data_cols = [c for c in fmt_cols if c not in IDENTITY_KEEP]
    stale_fmt = res['announce_date'] < res['prev_q']
    res.loc[stale_fmt, fmt_data_cols] = np.nan

    # ---------- actuals: actuals_announce_date <= 调仓日 最近一条 ----------
    act = (df[['security_id'] + actuals_cols]
           .dropna(subset=['actuals_announce_date'])
           .sort_values('actuals_announce_date')
           .drop_duplicates(['security_id', 'actuals_announce_date'], keep='last'))
    res = pd.merge_asof(res.sort_values('rebalance_date'), act.sort_values('actuals_announce_date'),
                        by='security_id', left_on='rebalance_date', right_on='actuals_announce_date',
                        direction='backward')
    # 守卫:actuals_announce_date 早于上一调仓日 -> actuals 过期,全置 NaN
    stale_act = res['actuals_announce_date'] < res['prev_q']
    res.loc[stale_act, actuals_cols] = np.nan

    # ---------- consensus: 期P 相同 且 statistic_date <= A 最后一条 ----------
    cons = (df[['security_id'] + consensus_cols]
            .dropna(subset=['consensus_forecast_period_end', 'consensus_statistic_date'])
            .drop_duplicates(['security_id', 'consensus_forecast_period_end',
                              'consensus_statistic_date'], keep='last').copy())
    cons['_pe'] = cons['consensus_forecast_period_end'].astype('datetime64[ns]')

    left = res[['security_id', 'rebalance_date', 'actuals_period_end', 'actuals_announce_date']].copy()
    left['_pe'] = left['actuals_period_end'].astype('datetime64[ns]')
    left_ok = left.dropna(subset=['actuals_period_end', 'actuals_announce_date'])

    cm = pd.merge_asof(
        left_ok.sort_values('actuals_announce_date'),
        cons.sort_values('consensus_statistic_date'),
        by=['security_id', '_pe'],
        left_on='actuals_announce_date', right_on='consensus_statistic_date',
        direction='backward', allow_exact_matches=True)   # <= A(公布当天或之前)

    res = res.merge(cm[['security_id', 'rebalance_date'] + consensus_cols],
                    on=['security_id', 'rebalance_date'], how='left')

    # 守卫:对应 actual 的 EPS 为 NaN(过期/缺失)-> consensus 全置 NaN
    res.loc[res['actuals_actual_eps'].isna(), consensus_cols] = np.nan

    return (res.drop(columns=['prev_q'])
               .sort_values(['security_id', 'rebalance_date']).reset_index(drop=True))

In [11]:
panel = build_rebalance_panel(df_quarterly)
print("总行数:", len(panel))
print("FMT 过期为空比例:", panel['total_assets'].isna().mean().round(3))
print("actuals 过期为空比例:", panel['actuals_actual_eps'].isna().mean().round(3))

总行数: 66349
FMT 过期为空比例: 0.01
actuals 过期为空比例: 0.012


In [12]:
panel

,security_id,rebalance_date,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,consensus_eps_mean,consensus_eps_median,consensus_estimate_dispersion,consensus_estimate_high,consensus_estimate_low,consensus_days_since_actual_eps,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag
0,10001,2011-03-31,EGAS,012994,NaT,NaT,NaN,NaN,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2011-06-30,EGAS,012994,2011-03-31,2011-05-11,2011.0,1.0,55,551020,...,0.48,0.48,0.00,0.48,0.48,7.0,True,True,False,False
2,10001,2011-09-30,EGAS,012994,2011-06-30,2011-08-12,2011.0,2.0,55,551020,...,0.01,0.01,0.00,0.01,0.01,5.0,True,True,False,False
3,10001,2011-12-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
4,10001,2012-03-31,EGAS,012994,NaT,NaT,NaN,NaN,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66344,93436,2023-12-31,TSLA,184996,2023-09-30,2023-10-18,2023.0,3.0,25,251020,...,0.80,0.80,0.09,1.02,0.65,0.0,False,False,False,False
66345,93436,2024-03-31,TSLA,184996,2023-12-31,2024-01-24,2023.0,4.0,25,251020,...,0.74,0.74,0.08,0.92,0.55,21.0,True,False,False,False
66346,93436,2024-06-30,TSLA,184996,2024-03-31,2024-04-23,2024.0,1.0,25,251020,...,0.51,0.45,0.17,0.87,0.20,22.0,True,False,False,False
66347,93436,2024-09-30,TSLA,184996,2024-06-30,2024-07-23,2024.0,2.0,25,251020,...,0.62,0.62,0.11,0.87,0.42,22.0,True,False,False,False


In [13]:
def apply_dnu_and_drop(df):
    df = df.copy()
    dnu_flags = [c for c in df.columns if c.endswith('_dnu_flag')]

    # flag 对应主列 + 连带置空的相关列
    special = {
        'ocf':   ['operating_cash_flow_ytd', 'ocf_increment'],   # ocf_dnu_flag
        'capex': ['capex_ytd', 'capex_increment'],               # capex_dnu_flag
    }

    report = {}
    for flag in dnu_flags:
        base = flag[:-len('_dnu_flag')]
        if base in special:                          # ocf / capex 特例(含连带列)
            targets = [c for c in special[base] if c in df.columns]
        elif base in df.columns:                     # 常规:X_dnu_flag -> X
            targets = [base]
        else:                                        # 兜底
            targets = [c for c in df.columns
                       if c.startswith(base + '_') and not c.endswith('_flag')]
        mask = (df[flag] == True)                     # NaN 当 False,不置空
        if targets and mask.any():
            df.loc[mask, targets] = np.nan
        report[flag] = (targets, int(mask.sum()))

    df = df.drop(columns=dnu_flags)                    # 删掉所有 _dnu_flag
    return df, report

In [14]:
panel, dnu_report = apply_dnu_and_drop(panel)
for k, v in dnu_report.items():
    print(f"{k:35s} -> {v[0]}  命中 {v[1]}")

book_equity_dnu_flag                -> ['book_equity']  命中 0
capex_dnu_flag                      -> ['capex_ytd', 'capex_increment']  命中 3588
cash_and_equivalents_dnu_flag       -> ['cash_and_equivalents']  命中 0
cogs_dnu_flag                       -> ['cogs']  命中 3
common_equity_dnu_flag              -> ['common_equity']  命中 31
current_liabilities_dnu_flag        -> ['current_liabilities']  命中 11223
eps_basic_dnu_flag                  -> ['eps_basic']  命中 3
eps_diluted_dnu_flag                -> ['eps_diluted']  命中 3
gross_profit_dnu_flag               -> ['gross_profit']  命中 2170
income_taxes_dnu_flag               -> ['income_taxes']  命中 3
interest_expense_dnu_flag           -> ['interest_expense']  命中 6749
inventory_dnu_flag                  -> ['inventory']  命中 1288
long_term_debt_dnu_flag             -> ['long_term_debt']  命中 28
net_income_dnu_flag                 -> ['net_income']  命中 3
net_ppe_dnu_flag                    -> ['net_ppe']  命中 3248
ocf_dnu_flag                      

In [15]:
panel

,security_id,rebalance_date,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,consensus_eps_mean,consensus_eps_median,consensus_estimate_dispersion,consensus_estimate_high,consensus_estimate_low,consensus_days_since_actual_eps,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag
0,10001,2011-03-31,EGAS,012994,NaT,NaT,NaN,NaN,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,2011-06-30,EGAS,012994,2011-03-31,2011-05-11,2011.0,1.0,55,551020,...,0.48,0.48,0.00,0.48,0.48,7.0,True,True,False,False
2,10001,2011-09-30,EGAS,012994,2011-06-30,2011-08-12,2011.0,2.0,55,551020,...,0.01,0.01,0.00,0.01,0.01,5.0,True,True,False,False
3,10001,2011-12-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
4,10001,2012-03-31,EGAS,012994,NaT,NaT,NaN,NaN,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66344,93436,2023-12-31,TSLA,184996,2023-09-30,2023-10-18,2023.0,3.0,25,251020,...,0.80,0.80,0.09,1.02,0.65,0.0,False,False,False,False
66345,93436,2024-03-31,TSLA,184996,2023-12-31,2024-01-24,2023.0,4.0,25,251020,...,0.74,0.74,0.08,0.92,0.55,21.0,True,False,False,False
66346,93436,2024-06-30,TSLA,184996,2024-03-31,2024-04-23,2024.0,1.0,25,251020,...,0.51,0.45,0.17,0.87,0.20,22.0,True,False,False,False
66347,93436,2024-09-30,TSLA,184996,2024-06-30,2024-07-23,2024.0,2.0,25,251020,...,0.62,0.62,0.11,0.87,0.42,22.0,True,False,False,False


In [16]:
panel.to_parquet("fmt_quarterly_panel.parquet", index=False)

In [61]:
df_d = pd.read_parquet("daily_panel.parquet")

In [62]:
df_d

,date,security_id,ticker,company_id,gics_sector,gics_industry,gics_subindustry,equity_ticker,equity_company_name_pit,equity_share_class_code,...,equity_adj_open,equity_adj_high,equity_adj_low,equity_adj_close,equity_adj_volume,equity_total_return,equity_price_return_ex_div,equity_shares_outstanding_k,equity_price_adj_factor,equity_share_adj_factor
0,2011-01-20,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.79,10.7900,10.6800,10.7600,20600.0,0.002796,0.002796,7834.0,1.0,1.0
1,2011-01-21,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.81,10.8100,10.5500,10.6699,21100.0,-0.008374,-0.008374,7834.0,1.0,1.0
2,2011-01-24,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.74,10.8800,10.7201,10.7700,20100.0,0.009382,0.009382,7834.0,1.0,1.0
3,2011-01-25,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.81,10.8699,10.7850,10.8699,22000.0,0.009276,0.009276,7834.0,1.0,1.0
4,2011-01-26,10001,EGAS,012994,55,551020,55102010,EGAS,GAS NATURAL INC,11.0,...,10.91,10.9200,10.8000,10.9200,30300.0,0.004609,0.004609,7834.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4131408,2024-12-24,93436,TSLA,184996,25,251020,25102010,TSLA,TESLA INC,11.0,...,435.90,462.7800,435.1400,462.2800,59351506.0,0.073572,0.073572,3210060.0,1.0,1.0
4131409,2024-12-26,93436,TSLA,184996,25,251020,25102010,TSLA,TESLA INC,11.0,...,465.16,465.3299,451.0200,454.1300,76392273.0,-0.017630,-0.017630,3210060.0,1.0,1.0
4131410,2024-12-27,93436,TSLA,184996,25,251020,25102010,TSLA,TESLA INC,11.0,...,449.52,450.0000,426.5000,431.6600,82370345.0,-0.049479,-0.049479,3210060.0,1.0,1.0
4131411,2024-12-30,93436,TSLA,184996,25,251020,25102010,TSLA,TESLA INC,11.0,...,419.40,427.0000,415.7500,417.4100,64705452.0,-0.033012,-0.033012,3210060.0,1.0,1.0


In [63]:
df_m = pd.read_parquet("monthly_panel.parquet")

In [64]:
df_m

,security_id,ticker,date,rec_mean,rec_median,rec_stdev,rec_count,rec_buypct,rec_holdpct,rec_sellpct,...,tp_target_price_high,tp_target_price_low,tp_target_price_stdev,tp_single_analyst_flag,tp_zero_price_flag,macro_cpi_index,macro_cpi_yoy,macro_unemployment_rate,macro_industrial_production,macro_manuf_employment_k
0,10001,EGAS,2011-01-31,1.00,1.0,0.00,1.0,100.00,0.00,0.00,...,12.0,12.00,NaN,True,False,220.472,1.437793,9.3,93.6787,11579
1,10001,EGAS,2011-02-28,1.00,1.0,0.00,1.0,100.00,0.00,0.00,...,12.0,12.00,NaN,True,False,221.187,1.700783,9.1,93.4814,11606
2,10001,EGAS,2011-03-31,1.00,1.0,0.00,1.0,100.00,0.00,0.00,...,12.0,12.00,NaN,True,False,221.898,2.124898,9.0,93.1102,11637
3,10001,EGAS,2011-04-29,1.00,1.0,0.00,1.0,100.00,0.00,0.00,...,12.5,12.50,NaN,True,False,223.046,2.619242,9.0,94.0788,11659
4,10001,EGAS,2011-05-31,1.00,1.0,0.00,1.0,100.00,0.00,0.00,...,12.5,12.50,NaN,True,False,224.093,3.077234,9.1,93.7749,11689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197990,93436,TSLA,2024-08-30,2.84,3.0,1.09,50.0,36.00,40.00,24.00,...,310.0,24.86,69.513,False,False,313.569,2.941476,4.2,99.9757,12806
197991,93436,TSLA,2024-09-30,2.77,3.0,1.10,53.0,39.62,37.74,22.64,...,310.0,24.86,68.498,False,False,314.062,2.607144,4.2,100.4309,12772
197992,93436,TSLA,2024-10-31,2.79,3.0,1.12,53.0,39.62,35.85,24.53,...,310.0,24.86,67.922,False,False,314.732,2.426483,4.1,99.8084,12758
197993,93436,TSLA,2024-11-29,2.74,3.0,1.14,54.0,42.59,33.33,24.07,...,400.0,24.86,76.412,False,False,315.631,2.578844,4.1,99.4695,12697


In [66]:
df_q = pd.read_parquet("fmt_quarterly_panel.parquet")

In [67]:
df_q

,security_id,date,ticker,company_id,report_date,announce_date,fiscal_year,fiscal_quarter,gics_sector,gics_industry,...,consensus_eps_mean,consensus_eps_median,consensus_estimate_dispersion,consensus_estimate_high,consensus_estimate_low,consensus_days_since_actual_eps,consensus_stale_consensus_flag,consensus_single_analyst_flag,consensus_eps_outlier_flag,consensus_forward_looking_flag
0,10001,2011-03-31,EGAS,012994,2010-09-30,2010-11-15,2010.0,3.0,55,551020,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
1,10001,2011-06-30,EGAS,012994,2011-03-31,2011-05-11,2011.0,1.0,55,551020,...,0.48,0.48,0.00,0.48,0.48,7.0,True,True,False,False
2,10001,2011-09-30,EGAS,012994,2011-06-30,2011-08-12,2011.0,2.0,55,551020,...,0.01,0.01,0.00,0.01,0.01,5.0,True,True,False,False
3,10001,2011-12-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
4,10001,2012-03-31,EGAS,012994,2011-09-30,2011-11-14,2011.0,3.0,55,551020,...,-0.11,-0.11,0.00,-0.11,-0.11,2.0,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66344,93436,2023-12-31,TSLA,184996,2023-09-30,2023-10-18,2023.0,3.0,25,251020,...,0.80,0.80,0.09,1.02,0.65,0.0,False,False,False,False
66345,93436,2024-03-31,TSLA,184996,2023-12-31,2024-01-24,2023.0,4.0,25,251020,...,0.74,0.74,0.08,0.92,0.55,21.0,True,False,False,False
66346,93436,2024-06-30,TSLA,184996,2024-03-31,2024-04-23,2024.0,1.0,25,251020,...,0.51,0.45,0.17,0.87,0.20,22.0,True,False,False,False
66347,93436,2024-09-30,TSLA,184996,2024-06-30,2024-07-23,2024.0,2.0,25,251020,...,0.62,0.62,0.11,0.87,0.42,22.0,True,False,False,False


In [ ]:
"""
factor_utils.py —— 全项目共用的因子中间算法(Section 1: Shared Raw Variables)
三种频率(daily / monthly / quarterly)的所有 alpha 都可调用。

约定:
- by    : 实体键,默认 'security_id'
- order : 时间排序键。季度面板用 'rebalance_date',日/月频面板用 'date'
所有返回值都是与输入 df 行对齐的 pd.Series。

★重要前提(相对旧版的改动):
  build_rebalance_panel 的过期守卫已保证季度面板每个季度槽位只有一条
  (缺报的是 NaN 行,不是重复行),因此季度函数不再按 report_date 去重,
  统一按行 shift。固定每年 4 行的结构让 lag_4q(shift 4 行)天然对齐去年同期,
  缺季的 NaN 行作为占位保留,不会使 YoY 错位。
  -> 季度 / 月频 / 日频全部用同一个 group_shift,只是 periods 和 order 不同。
"""
import numpy as np
import pandas as pd

# ======================================================================
# 各频率 alpha 用到的 func 速查(func 可重复;★=特殊用法)
# ----------------------------------------------------------------------
# 【季度 Quarterly · target_63d · 季末采样】 df_q (order='rebalance_date')
#   rolling_sum_4q(TTM) · lag_1q · lag_4q · safe_divide · score/score_low
#  
# 【日频价格挂钩 · target_21d · 月末筛选】 df_d (order='date')
#   market_cap_mn · rolling_sum_4q(季度面板算好,再对齐日频) · safe_divide · score/score_low
#
# 【月频 · target_21d · 月末采样】 df_m (order='date')
#   group_shift(periods=1或3) · safe_divide · score/score_low
#   ★ RatingChange_1M/3M 必须 group_shift(df_m,'rec_mean',1或3);✗ 别用 lag_21d/63d
#
# 【复合 Composites】 score/score_low(横截面 by=采样日) + 各成分最终值
# ======================================================================


# ======================================================================
# 一、通用分组滞后 group_shift(季度/月/日频都用它)
# ======================================================================
def group_shift(df, col, periods, by='security_id', order='date'):
    """按 by 分组、order 排序后 shift periods 行,结果对齐回原 index。"""
    d = df.sort_values([by, order])
    return d.groupby(by, sort=False)[col].shift(periods).reindex(df.index)


# ======================================================================
# 二、季度类:lag_1q / lag_4q / rolling_sum_4q(TTM)
#    直接按行 shift(order='rebalance_date');前提见文件头 ★。
# ======================================================================
def lag_nq(df, col, n, by='security_id', order='rebalance_date'):
    """x 的 n 期前季度值(按行)。"""
    return group_shift(df, col, n, by, order)


def lag_1q(df, col, by='security_id', order='rebalance_date'):
    """上一季度值(QoQ 用)。"""
    return group_shift(df, col, 1, by, order)


def lag_4q(df, col, by='security_id', order='rebalance_date'):
    """四个季度前的值(YoY 用,同一财季去年同期)。"""
    return group_shift(df, col, 4, by, order)


def rolling_sum_4q(df, col, by='security_id', order='rebalance_date', min_valid=3):
    """TTM:最近四个季度之和。
    - 公司历史不足 4 个季度(组内前 3 行)-> NaN(窗口还没满 4 季);
    - 已满 4 季、缺 1 个洞 -> 用可得季度均值补(等价 mean(有效)*4);
    - 缺 2 个及以上(有效 < min_valid=3)-> NaN。
    """
    d = df.sort_values([by, order])
    g = d.groupby(by, sort=False)
    m = (g[col].rolling(4, min_periods=min_valid).mean()
              .reset_index(level=0, drop=True)) * 4
    m = m.where(g.cumcount() >= 3)          # 组内不足 4 季(前3行)-> NaN
    return m.reindex(df.index)


# TTM 别名
ttm = rolling_sum_4q


# ======================================================================
# 三、日/月频滞后:lag_21d / lag_63d(日频面板;月频用 group_shift(periods=1/3))
# ======================================================================
def lag_21d(df, col, by='security_id', order='date'):
    """21 个交易日前(约 1 个月,日频面板用)。"""
    return group_shift(df, col, 21, by, order)


def lag_63d(df, col, by='security_id', order='date'):
    """63 个交易日前(约 3 个月,日频面板用)。"""
    return group_shift(df, col, 63, by, order)


# ======================================================================
# 四、派生量:market_cap_mn / avg_*(流量对存量的期初期末平均)
# ======================================================================
def market_cap_mn(df, close='equity_close_raw', shares_k='equity_shares_outstanding_k'):
    """市值(百万):close * shares(千股) / 1000。"""
    return df[close] * df[shares_k] / 1000.0


def avg_total_assets(df, by='security_id', order='rebalance_date'):
    return 0.5 * (df['total_assets'] + lag_4q(df, 'total_assets', by, order))


def avg_book_equity(df, by='security_id', order='rebalance_date'):
    return 0.5 * (df['book_equity'] + lag_4q(df, 'book_equity', by, order))


def avg_inventory(df, by='security_id', order='rebalance_date'):
    return 0.5 * (df['inventory'] + lag_4q(df, 'inventory', by, order))


# ======================================================================
# 五、除法安全:Zero numerator rule
#    分母为 0 / 无效 -> NaN;零分子(debt/PPE/capex=0)有经济意义,保留为 0。
# ======================================================================
def safe_divide(num, den):
    """num/den;den==0 或无效 -> NaN。零分子自然得 0(保留)。"""
    num = pd.Series(np.asarray(num, dtype='float64'),
                    index=num.index if isinstance(num, pd.Series) else None)
    den = pd.Series(np.asarray(den, dtype='float64'),
                    index=den.index if isinstance(den, pd.Series) else None)
    den = den.where(den != 0, np.nan)              # 分母 0 -> 无效
    return num / den


# ======================================================================
# 六、打分:score / score_low(横截面 winsorize + 百分位排名,复合因子内部用)
#    score(x) = percentile_rank(winsorize(x,1%,99%)) - 0.5,越高越好。
#    score_low(x) = score(-x),用于杠杆/离散度等"越低越好"的因子。
# ======================================================================
def winsorize(s, lower=0.01, upper=0.99):
    lo, hi = s.quantile(lower), s.quantile(upper)
    return s.clip(lo, hi)


def _xs_score(s):
    return winsorize(s).rank(pct=True) - 0.5


def score(df, col, by='rebalance_date'):
    """横截面(按 by,默认采样日)百分位打分。"""
    return df.groupby(by, sort=False)[col].transform(_xs_score)


def score_low(df, col, by='rebalance_date'):
    """score(-x):越低越好。"""
    neg = (-df[col])
    return neg.groupby(df[by], sort=False).transform(_xs_score)